In [7]:
import os
import sys
import time
import subprocess
from pathlib import Path


def find_ai_backend_dir() -> Path | None:
    cwd = Path.cwd()

    # 1) current dir
    if (cwd / "main.py").exists() and (cwd / "qwen3_tts.py").exists():
        return cwd

    # 2) ./ai_backend
    if (cwd / "ai_backend" / "main.py").exists() and (cwd / "ai_backend" / "qwen3_tts.py").exists():
        return cwd / "ai_backend"

    # 3) common Colab-style paths
    search_roots = [cwd, Path("/content"), Path("/content/drive/MyDrive")]
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("main.py"):
            if p.parent.joinpath("qwen3_tts.py").exists():
                return p.parent

    return None


ai_dir = find_ai_backend_dir()
print("Kernel cwd:", Path.cwd())

if ai_dir is None:
    print("❌ Could not find folder containing both main.py and qwen3_tts.py")
    print("Please upload/mount the project in this kernel, then re-run this cell.")
    print("Expected folder example: /content/MANJU-frontend/ai_backend")
else:
    os.chdir(ai_dir)
    print("✅ Using directory:", Path.cwd())

    # Stop previously started processes (if re-running this cell)
    for var_name in ["MAIN_PROC", "QWEN_PROC"]:
        proc = globals().get(var_name)
        if proc and proc.poll() is None:
            print(f"Stopping previous {var_name} (PID {proc.pid})...")
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()

    python_exe = sys.executable

    MAIN_PROC = subprocess.Popen(
        [python_exe, "main.py"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    QWEN_PROC = subprocess.Popen(
        [python_exe, "qwen3_tts.py"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    time.sleep(2)

    print(f"main.py PID: {MAIN_PROC.pid}, running: {MAIN_PROC.poll() is None}")
    print(f"qwen3_tts.py PID: {QWEN_PROC.pid}, running: {QWEN_PROC.poll() is None}")

    if MAIN_PROC.poll() is not None or QWEN_PROC.poll() is not None:
        print("⚠️ One or both processes exited quickly. Check dependencies/env in this kernel.")
    else:
        print("✅ Both services are running.")

    print("Re-run this cell to restart both services.")

Kernel cwd: /content
❌ Could not find folder containing both main.py and qwen3_tts.py
Please upload/mount the project in this kernel, then re-run this cell.
Expected folder example: /content/MANJU-frontend/ai_backend
